In [1]:
import subprocess, sys
needs_restart = False
for pkg_name, import_name in [('xgboost', 'xgboost'), ('lightgbm', 'lightgbm')]:
    try:
        __import__(import_name)
        print(f'{import_name} ya disponible ✅')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               pkg_name, '--quiet'])
        needs_restart = True
        print(f'{pkg_name} instalado ✅ — reiniciar kernel')
if needs_restart:
    print('\n⚠️  REINICIAR KERNEL antes de continuar (Cell 0).')
else:
    print('\nTodas las dependencias disponibles.')


xgboost ya disponible ✅
lightgbm instalado ✅ — reiniciar kernel

⚠️  REINICIAR KERNEL antes de continuar (Cell 0).



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
import os, gc, warnings, time, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                              confusion_matrix, cohen_kappa_score)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit

import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SAVE_DIR     = '/srv/work'
IN_FEAT      = f'{SAVE_DIR}/te_features_v5.parquet'
OUT_DIR      = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':         {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'mae_ord':     {'floor': 0.55, 'target': 0.45, 'stretch': 0.35},
    'macro_f1':    {'floor': 0.55, 'target': 0.65, 'stretch': 0.75},
    'bal_acc':     {'floor': 0.60, 'target': 0.70, 'stretch': 0.78},
    'recall_alta': {'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
    'prec_alta':   {'floor': 0.50, 'target': 0.62, 'stretch': 0.75},
}

import sys
if '/srv/python_packages' not in sys.path:
    sys.path.insert(0, '/srv/python_packages')

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

MLFLOW_URI        = 'http://localhost:6000'
MLFLOW_EXPERIMENT = 'latam2cloud'

try:
    import urllib.request
    urllib.request.urlopen(f'{MLFLOW_URI}/health', timeout=5)
    print(f'MLflow OK — {MLFLOW_URI}')
except Exception as e:
    print(f'[ADVERTENCIA] MLflow no responde: {e}')
    print('  Abrir terminal: bash /srv/start_mlflow.sh')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f'NumPy {np.__version__}  |  Pandas {pd.__version__}  |  '
      f'XGBoost {xgb.__version__}  |  LightGBM {lgb.__version__}  |  '
      f'TensorFlow {tf.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}/')
print('\nCriterios H2 (congelados — Benchmarking v2.0):')
for m, t in UMBRALES.items():
    print(f'  {m:<14}: Floor={t["floor"]}  Target={t["target"]}  Stretch={t["stretch"]}')


MLflow OK — http://localhost:6000
NumPy 1.26.4  |  Pandas 3.0.3  |  XGBoost 3.2.0  |  LightGBM 4.7.0  |  TensorFlow 2.19.1
Input : /srv/work/te_features_v5.parquet
Output: /srv/work/models/

Criterios H2 (congelados — Benchmarking v2.0):
  qwk           : Floor=0.65  Target=0.72  Stretch=0.8
  mae_ord       : Floor=0.55  Target=0.45  Stretch=0.35
  macro_f1      : Floor=0.55  Target=0.65  Stretch=0.75
  bal_acc       : Floor=0.6  Target=0.7  Stretch=0.78
  recall_alta   : Floor=0.55  Target=0.75  Stretch=0.85
  prec_alta     : Floor=0.5  Target=0.62  Stretch=0.75


In [3]:

print('Cargando te_features_v5.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'  Shape: {df.shape}')

df = df.sort_values(ROUTE_KEYS + ['window'])
df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift: {df.shape}')

feat_cols_v5 = [c for c in df.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]
print(f'\nFeatures modelo v5: {len(feat_cols_v5)}')

new_feats_v5 = ['y_lag1','y_lag2','y_lag3','streak_length',
                'd_latency','d_jitter','d_loss','latency_ratio_lag48']
for f in new_feats_v5:
    assert f in feat_cols_v5, f'ERROR: falta {f} en feat_cols_v5'
print(f'  Features nuevas v5 presentes: {new_feats_v5} ✅')

df_imp = df.copy()
ratio_cols = [c for c in feat_cols_v5 if 'ratio' in c]
df_imp[ratio_cols] = df_imp[ratio_cols].fillna(1.0)
other_nan  = [c for c in feat_cols_v5 if c not in ratio_cols]
df_imp[other_nan] = df_imp[other_nan].fillna(0.0)

std_cols = [c for c in feat_cols_v5 if c.startswith('std_')]
bgp_cols = [c for c in feat_cols_v5 if c.startswith('bgp_')]
df_xgb = df.copy()
df_xgb[std_cols + bgp_cols] = df_xgb[std_cols + bgp_cols].fillna(0.0)

print('D17-v5 aplicado:')
print('  df_imp : NaN->0 general; latency_ratio_lag48->1.0  (RF, GRU)')
print('  df_xgb : NaN nativo en lags/ratio; fillna(0) en std_* y bgp_* (XGB, LGB)')

train_imp = df_imp[df_imp[SPLIT_COL]=='train'].reset_index(drop=True)
val_imp   = df_imp[df_imp[SPLIT_COL]=='val'].reset_index(drop=True)
train_xgb = df_xgb[df_xgb[SPLIT_COL]=='train'].reset_index(drop=True)
val_xgb   = df_xgb[df_xgb[SPLIT_COL]=='val'].reset_index(drop=True)

X_train_raw = train_imp[feat_cols_v5].values.astype('float32')
y_train     = train_imp['y'].values.astype(int)
X_val_raw   = val_imp[feat_cols_v5].values.astype('float32')
y_val       = val_imp['y'].values.astype(int)

X_train_xgb = train_xgb[feat_cols_v5].values.astype('float32')
X_val_xgb   = val_xgb[feat_cols_v5].values.astype('float32')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc   = scaler.transform(X_val_raw)
with open(f'{OUT_DIR}/scaler_v5.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'\nScaler v5 guardado: {OUT_DIR}/scaler_v5.pkl')

classes      = np.array([0,1,2,3])
class_counts = np.bincount(y_train)
n_total      = len(y_train)
sqrt_w_arr   = np.sqrt(n_total / (4 * class_counts)).astype('float32')
sqrt_w_dict  = dict(zip(classes, sqrt_w_arr))
sample_weights    = np.array([sqrt_w_dict[y] for y in y_train], dtype='float32')
class_weight_dict = {int(c): float(w) for c, w in sqrt_w_dict.items()}

print(f'\nDistribucion y_train:')
for c in range(4):
    pct = class_counts[c]/n_total
    print(f'  {LABEL_COLS[c]:<8}: {class_counts[c]:>8,} ({pct:.1%})  peso_sqrt={sqrt_w_arr[c]:.3f}')
print(f'\nSplits: train={len(y_train):,}  val={len(y_val):,}')


Cargando te_features_v5.parquet...
  Shape: (3763917, 109)
  Post target-shift: (3761847, 110)

Features modelo v5: 100
  Features nuevas v5 presentes: ['y_lag1', 'y_lag2', 'y_lag3', 'streak_length', 'd_latency', 'd_jitter', 'd_loss', 'latency_ratio_lag48'] ✅
D17-v5 aplicado:
  df_imp : NaN->0 general; latency_ratio_lag48->1.0  (RF, GRU)
  df_xgb : NaN nativo en lags/ratio; fillna(0) en std_* y bgp_* (XGB, LGB)

Scaler v5 guardado: /srv/work/models/scaler_v5.pkl

Distribucion y_train:
  Normal  : 1,899,347 (89.3%)  peso_sqrt=0.529
  Baja    :  153,319 (7.2%)  peso_sqrt=1.862
  Media   :   36,421 (1.7%)  peso_sqrt=3.820
  Alta    :   36,691 (1.7%)  peso_sqrt=3.806

Splits: train=2,125,778  val=644,690


In [4]:

def mae_ordinal(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def eval_model(y_true, y_pred, label=''):
    """
    Metricas: QWK, MacroF1, BalAcc, MAEord, RecAlta, PrecAlta.
    QWK: Cohen (1968, Psych. Bull. 70:213-220); Landis & Koch (1977, Biometrics 33:159-174).
    H2 candidatura: QWK>=0.65 AND RecAlta>=0.55 (Benchmarking v2.0 congelado).
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    qwk     = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    macro_f1= f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mae_ord = mae_ordinal(y_true, y_pred)
    cm      = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    tp_alta = cm[3,3]
    fn_alta = cm[3,:].sum() - tp_alta
    fp_alta = cm[:,3].sum() - tp_alta
    rec_alta = tp_alta / (tp_alta + fn_alta + 1e-9)
    prec_alta= tp_alta / (tp_alta + fp_alta + 1e-9)
    candidato= bool(qwk >= UMBRALES['qwk']['floor'] and
                    rec_alta >= UMBRALES['recall_alta']['floor'])
    W = 110
    print(f'\n{"═"*W}')
    print(f'  {label}')
    print(f'  QWK={qwk:.4f}  MacroF1={macro_f1:.4f}  BalAcc={bal_acc:.4f}'
          f'  MAEord={mae_ord:.4f}  RecAlta={rec_alta:.4f}  PrecAlta={prec_alta:.4f}')
    cand_str = '✅ CANDIDATO H2' if candidato else '❌ No candidato'
    print(f'  {cand_str}  (QWK≥0.65={qwk>=0.65}  RecAlta≥0.55={rec_alta>=0.55})')
    print(f'{"═"*W}')
    print(f'  Confusion matrix (filas=real, cols=pred):')
    print(f'  {"":8s}  Normal   Baja    Media   Alta')
    for i, lbl in enumerate(['Normal','Baja','Media','Alta']):
        print(f'  {lbl:<8s}  {cm[i,0]:>6,}  {cm[i,1]:>6,}  {cm[i,2]:>6,}  {cm[i,3]:>6,}')
    return dict(qwk=qwk, macro_f1=macro_f1, bal_acc=bal_acc, mae_ord=mae_ord,
                recall_alta=rec_alta, prec_alta=prec_alta, candidato_crit=candidato,
                confusion_matrix=cm)

resultados_v4_ref = {
    'baseline_v4_val'    : dict(qwk=0.6149, recall_alta=0.5598, candidato_crit=False),
    'rf_v4_full_val'     : dict(qwk=0.6257, recall_alta=0.6466, candidato_crit=False),
    'rf_v4_sind12_val'   : dict(qwk=0.6296, recall_alta=0.6516, candidato_crit=False),
    'xgb_v4_val'         : dict(qwk=0.6141, recall_alta=0.6941, candidato_crit=False),
    'gru_v4_val'         : dict(qwk=0.4996, recall_alta=0.0000, candidato_crit=False),
    'rf_v4_sind12_test'  : dict(qwk=0.6101, recall_alta=0.5815, candidato_crit=False),
}
resultados_v5 = {}

print('Funciones de evaluacion listas.')
print('\nReferencias v4 (val):')
print(f'  {"Modelo":<22}  QWK     RecAlta  Candidato')
for k, r in resultados_v4_ref.items():
    if 'val' in k:
        cand = '✅' if r['candidato_crit'] else '❌'
        print(f'  {k:<22}  {r["qwk"]:.4f}  {r["recall_alta"]:.4f}   {cand}')
print('\nGap al floor H2: QWK necesita +{:.4f} sobre mejor v4 (RF Sin-D12 val=0.6296)'.format(
    0.65 - 0.6296))


Funciones de evaluacion listas.

Referencias v4 (val):
  Modelo                  QWK     RecAlta  Candidato
  baseline_v4_val         0.6149  0.5598   ❌
  rf_v4_full_val          0.6257  0.6466   ❌
  rf_v4_sind12_val        0.6296  0.6516   ❌
  xgb_v4_val              0.6141  0.6941   ❌
  gru_v4_val              0.4996  0.0000   ❌

Gap al floor H2: QWK necesita +0.0204 sobre mejor v4 (RF Sin-D12 val=0.6296)


In [5]:
y_pred_base = val_imp[TARGET_COL].values.astype(int)
resultados_v5['baseline_v5'] = eval_model(y_val, y_pred_base, 'Baseline Persistencia v5 — Validacion')
print('\nNote: baseline v5 ~= baseline v4 (mismo dataset, mismo protocolo)')
print(f'  v4 baseline val QWK=0.6149  |  v5 baseline val QWK={resultados_v5["baseline_v5"]["qwk"]:.4f}')



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Baseline Persistencia v5 — Validacion
  QWK=0.6149  MacroF1=0.6255  BalAcc=0.6254  MAEord=0.1655  RecAlta=0.5598  PrecAlta=0.5600
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Confusion matrix (filas=real, cols=pred):
            Normal   Baja    Media   Alta
  Normal    521,114  18,853   5,198   6,026
  Baja      18,910  36,552   2,149   1,034
  Media      5,282   2,205   5,509   1,773
  Alta       5,946     978   1,917  11,244

Note: baseline v5 ~= baseline v4 (mismo dataset, mismo protocolo)
  v4 baseline val QWK=0.6149  |  v5 baseline val QWK=0.6149


In [6]:
print(f'Entrenando RF v5 ({len(feat_cols_v5)} features, n_est=200)...')
t0 = time.time()

rf_v5 = RandomForestClassifier(
    n_estimators     = 200,
    max_depth        = 30,
    min_samples_leaf = 5,
    class_weight     = 'balanced',
    max_features     = 'sqrt',
    n_jobs           = 4,
    random_state     = RANDOM_STATE,
    verbose          = 1,
)
rf_v5.fit(X_train_sc, y_train)
elapsed = time.time() - t0
print(f'\nEntrenamiento: {elapsed/60:.1f} min')

y_pred_rf_v5 = rf_v5.predict(X_val_sc)
resultados_v5['rf_v5'] = eval_model(y_val, y_pred_rf_v5, 'RF v5 — Validacion')

importances_rf = pd.Series(rf_v5.feature_importances_, index=feat_cols_v5)
print('\nTop 25 features RF v5 (MDI):')
for feat, imp in importances_rf.sort_values(ascending=False).head(25).items():
    marker = ' <-- NEW v5' if feat in ['y_lag1','y_lag2','y_lag3','streak_length',
                                        'd_latency','d_jitter','d_loss','latency_ratio_lag48'] else ''
    print(f'  {feat:<42s} {imp:.4f}{marker}')

new_mdi = importances_rf[['y_lag1','y_lag2','y_lag3','streak_length',
                           'd_latency','d_jitter','d_loss','latency_ratio_lag48']].sum()
total_mdi = importances_rf.sum()
print(f'\nMDI features nuevas v5 acumulado: {new_mdi:.4f} ({new_mdi/total_mdi:.1%} del total)')

with open(f'{OUT_DIR}/rf_v5_model.pkl', 'wb') as f:
    pickle.dump(rf_v5, f)
print(f'\nModelo guardado: {OUT_DIR}/rf_v5_model.pkl')

r4 = resultados_v4_ref['rf_v4_sind12_val']
r5 = resultados_v5['rf_v5']
print('\nΔ RF v4-Sin-D12 -> RF v5 (val):')
for m in ['qwk','recall_alta','macro_f1','bal_acc']:
    delta = r5[m] - r4.get(m, 0)
    print(f'  {m:<14}: v4={r4.get(m,0):.4f}  v5={r5[m]:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='rf_v5'):
    mlflow.log_param('model_type',       'random_forest')
    mlflow.log_param('n_estimators',     200)
    mlflow.log_param('max_depth',        30)
    mlflow.log_param('min_samples_leaf', 5)
    mlflow.log_param('class_weight',     'balanced')
    mlflow.log_param('dataset_version',  'v5')
    mlflow.log_param('n_features',       len(feat_cols_v5))
    mlflow.log_param('d17_fillna',       'ratio->1.0_rest->0')
    mlflow.log_param('d28_n_est',        200)
    for m in ['qwk','recall_alta','macro_f1','bal_acc','mae_ord','prec_alta']:
        mlflow.log_metric(f'val_{m}', r5[m])
    mlflow.log_metric('val_candidato', int(r5['candidato_crit']))
    mlflow.log_param('model_local_path', f'{OUT_DIR}/rf_v5_model.pkl')
print('RF v5 registrado en MLflow.')


Entrenando RF v5 (100 features, n_est=200)...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  6.0min
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed: 26.0min
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed: 27.0min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.



Entrenamiento: 27.0 min


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   10.7s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:   11.2s finished



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  RF v5 — Validacion
  QWK=0.6420  MacroF1=0.6560  BalAcc=0.6720  MAEord=0.1594  RecAlta=0.6726  PrecAlta=0.5381
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Confusion matrix (filas=real, cols=pred):
            Normal   Baja    Media   Alta
  Normal    520,331  18,017   4,760   8,083
  Baja      16,261  39,146   1,824   1,414
  Media      4,782   1,925   5,962   2,100
  Alta       4,642     593   1,340  13,510

Top 25 features RF v5 (MDI):
  y_lag1                                     0.0543 <-- NEW v5
  streak_length                              0.0460 <-- NEW v5
  y_lag2                                     0.0406 <-- NEW v5
  y_lag3                                     0.0340 <-- NEW v5
  mean_loss                                  0.0284
  delta_p95_lat  

In [7]:
print('Entrenando XGBoost v5...')
print(f'  n_estimators=1500  |  lr=0.05  |  colsample=0.6  |  early_stop=50')

for _var in ['X_train_sc', 'X_val_sc', 'rf_v5']:
    if _var in globals():
        del globals()[_var]
gc.collect()

t0 = time.time()
xgb_v5 = xgb.XGBClassifier(
    n_estimators          = 1500,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.6,
    min_child_weight      = 5,
    gamma                 = 0.1,
    objective             = 'multi:softprob',
    num_class             = 4,
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 50,
    n_jobs                = 4,
    random_state          = RANDOM_STATE,
    verbosity             = 1,
    tree_method           = 'hist',
)
xgb_v5.fit(
    X_train_xgb, y_train,
    sample_weight = sample_weights,
    eval_set      = [(X_val_xgb, y_val)],
    verbose       = 100,
)
elapsed = time.time() - t0
print(f'\nTiempo: {elapsed/60:.1f} min  |  best_iteration: {xgb_v5.best_iteration}')

y_prob_xgb = xgb_v5.predict_proba(X_val_xgb)
y_pred_xgb = np.argmax(y_prob_xgb, axis=1)
resultados_v5['xgb_v5'] = eval_model(y_val, y_pred_xgb, 'XGBoost v5 — Validacion')

xgb_v5.get_booster().feature_names = feat_cols_v5
imp_xgb = (pd.Series(xgb_v5.get_booster().get_score(importance_type='gain'))
           .sort_values(ascending=False))
print('\nTop 15 features XGBoost v5 (gain):')
for feat, imp in imp_xgb.head(15).items():
    marker = ' <-- NEW v5' if feat in ['y_lag1','y_lag2','y_lag3','streak_length',
                                        'd_latency','d_jitter','d_loss','latency_ratio_lag48'] else ''
    print(f'  {feat:<42s} {imp:.2f}{marker}')

xgb_v5.save_model(f'{OUT_DIR}/xgb_v5_model.json')
print(f'\nModelo guardado: {OUT_DIR}/xgb_v5_model.json')

np.save(f'{OUT_DIR}/xgb_v5_val_proba.npy', y_prob_xgb)

r5 = resultados_v5['xgb_v5']
r4 = resultados_v4_ref['xgb_v4_val']
print('\nΔ XGB v4 -> XGB v5 (val):')
for m in ['qwk','recall_alta','macro_f1','bal_acc']:
    delta = r5[m] - r4.get(m, 0)
    print(f'  {m:<14}: v4={r4.get(m,0):.4f}  v5={r5[m]:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='xgb_v5'):
    mlflow.log_param('model_type',        'xgboost')
    mlflow.log_param('n_estimators_best', xgb_v5.best_iteration + 1)
    mlflow.log_param('max_depth',         6)
    mlflow.log_param('learning_rate',     0.05)
    mlflow.log_param('colsample_bytree',  0.6)
    mlflow.log_param('sample_weight',     'sqrt')
    mlflow.log_param('dataset_version',   'v5')
    mlflow.log_param('n_features',        len(feat_cols_v5))
    mlflow.log_param('d19_colsample',     0.6)
    mlflow.log_param('d28_n_est',         1500)
    for m in ['qwk','recall_alta','macro_f1','bal_acc','mae_ord','prec_alta']:
        mlflow.log_metric(f'val_{m}', r5[m])
    mlflow.log_metric('val_candidato', int(r5['candidato_crit']))
    mlflow.log_param('model_local_path',  f'{OUT_DIR}/xgb_v5_model.json')
print('XGBoost v5 registrado en MLflow.')


Entrenando XGBoost v5...
  n_estimators=1500  |  lr=0.05  |  colsample=0.6  |  early_stop=50
[0]	validation_0-mlogloss:0.64735
[100]	validation_0-mlogloss:0.33764
[200]	validation_0-mlogloss:0.33307
[300]	validation_0-mlogloss:0.32994
[400]	validation_0-mlogloss:0.32753
[500]	validation_0-mlogloss:0.32591
[600]	validation_0-mlogloss:0.32444
[700]	validation_0-mlogloss:0.32328
[800]	validation_0-mlogloss:0.32215
[900]	validation_0-mlogloss:0.32123
[1000]	validation_0-mlogloss:0.32033
[1100]	validation_0-mlogloss:0.31948
[1200]	validation_0-mlogloss:0.31869
[1300]	validation_0-mlogloss:0.31804
[1400]	validation_0-mlogloss:0.31734
[1499]	validation_0-mlogloss:0.31675

Tiempo: 23.7 min  |  best_iteration: 1499

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  XGBoost v5 — Validacion
  QWK=0.6232  MacroF1=0.6511  BalAcc=0.6985  MAEord=0.1759  RecAlta=0.6648  PrecAlta=0.5112
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)


In [8]:
print('Entrenando LightGBM v5 (D25)...')
print(f'  n_estimators=1000  |  lr=0.05  |  num_leaves=63  |  early_stop=50')

for _var in ['X_train_xgb', 'xgb_v5']:
    if _var in globals():
        del globals()[_var]
gc.collect()

dtrain_lgb = lgb.Dataset(
    train_xgb[feat_cols_v5].values.astype('float32'),
    label        = y_train,
    weight       = sample_weights,
    feature_name = feat_cols_v5,
    free_raw_data= False,
)
dval_lgb = lgb.Dataset(
    val_xgb[feat_cols_v5].values.astype('float32'),
    label        = y_val,
    feature_name = feat_cols_v5,
    reference    = dtrain_lgb,
    free_raw_data= False,
)

lgb_params = dict(
    objective         = 'multiclass',
    num_class         = 4,
    metric            = 'multi_logloss',
    num_leaves        = 63,
    max_depth         = -1,
    learning_rate     = 0.05,
    feature_fraction  = 0.6,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    min_child_samples = 20,
    min_child_weight  = 1e-3,
    n_jobs            = 4,
    random_state      = RANDOM_STATE,
    verbose           = -1,
)

t0 = time.time()
lgb_v5 = lgb.train(
    lgb_params,
    dtrain_lgb,
    num_boost_round    = 1000,
    valid_sets         = [dval_lgb],
    valid_names        = ['val'],
    callbacks          = [
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)
elapsed = time.time() - t0
print(f'\nTiempo: {elapsed/60:.1f} min  |  best_iteration: {lgb_v5.best_iteration}')

X_val_lgb  = val_xgb[feat_cols_v5].values.astype('float32')
y_prob_lgb = lgb_v5.predict(X_val_lgb, num_iteration=lgb_v5.best_iteration)
y_pred_lgb = np.argmax(y_prob_lgb, axis=1)
resultados_v5['lgb_v5'] = eval_model(y_val, y_pred_lgb, 'LightGBM v5 — Validacion')

imp_lgb = pd.Series(lgb_v5.feature_importance(importance_type='gain'),
                    index=lgb_v5.feature_name())
print('\nTop 15 features LightGBM v5 (gain):')
for feat, imp in imp_lgb.sort_values(ascending=False).head(15).items():
    marker = ' <-- NEW v5' if feat in ['y_lag1','y_lag2','y_lag3','streak_length',
                                        'd_latency','d_jitter','d_loss','latency_ratio_lag48'] else ''
    print(f'  {feat:<42s} {imp:.1f}{marker}')

lgb_v5.save_model(f'{OUT_DIR}/lgb_v5_model.txt')
np.save(f'{OUT_DIR}/lgb_v5_val_proba.npy', y_prob_lgb)
print(f'\nModelo guardado: {OUT_DIR}/lgb_v5_model.txt')

r5 = resultados_v5['lgb_v5']
with mlflow.start_run(run_name='lgb_v5'):
    mlflow.log_param('model_type',        'lightgbm')
    mlflow.log_param('n_estimators_best', lgb_v5.best_iteration)
    mlflow.log_param('num_leaves',        63)
    mlflow.log_param('learning_rate',     0.05)
    mlflow.log_param('feature_fraction',  0.6)
    mlflow.log_param('sample_weight',     'sqrt')
    mlflow.log_param('dataset_version',   'v5')
    mlflow.log_param('n_features',        len(feat_cols_v5))
    mlflow.log_param('d25_lightgbm',      True)
    for m in ['qwk','recall_alta','macro_f1','bal_acc','mae_ord','prec_alta']:
        mlflow.log_metric(f'val_{m}', r5[m])
    mlflow.log_metric('val_candidato', int(r5['candidato_crit']))
    mlflow.log_param('model_local_path',  f'{OUT_DIR}/lgb_v5_model.txt')
print('LightGBM v5 registrado en MLflow.')


Entrenando LightGBM v5 (D25)...
  n_estimators=1000  |  lr=0.05  |  num_leaves=63  |  early_stop=50
Training until validation scores don't improve for 50 rounds
[100]	val's multi_logloss: 0.333373
[200]	val's multi_logloss: 0.328335
[300]	val's multi_logloss: 0.325247
[400]	val's multi_logloss: 0.323069
[500]	val's multi_logloss: 0.321003
[600]	val's multi_logloss: 0.319267
[700]	val's multi_logloss: 0.317478
[800]	val's multi_logloss: 0.315887
[900]	val's multi_logloss: 0.314405
[1000]	val's multi_logloss: 0.312612
Did not meet early stopping. Best iteration is:
[1000]	val's multi_logloss: 0.312612

Tiempo: 10.3 min  |  best_iteration: 1000

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  LightGBM v5 — Validacion
  QWK=0.6300  MacroF1=0.6533  BalAcc=0.7025  MAEord=0.1745  RecAlta=0.6824  PrecAlta=0.5136
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
═════════════════════════════════════════════════════════════════

In [9]:

for _var in ['train_xgb','val_xgb','X_train_xgb','X_val_xgb',
             'X_val_lgb','dtrain_lgb','dval_lgb','lgb_v5',
             'train_imp','df_imp','df_xgb','df']:
    if _var in globals():
        del globals()[_var]
gc.collect()
print('Memoria liberada para GRU.')

SEQ_LEN       = 4
FEAT_GRU      = feat_cols_v5
MAX_SEQ_TRAIN = 400_000
MAX_SEQ_VAL   = 200_000

def build_sequences_labels_only(df_split, seq_len, feat_cols):
    """Phase 1: recolectar solo labels (bajo consumo de memoria)."""
    labels, positions = [], []
    grp_by = df_split.groupby(ROUTE_KEYS, sort=False)
    for key, grp in grp_by:
        grp_s = grp.sort_values('window')
        if len(grp_s) < seq_len + 1:
            continue
        y_arr = grp_s['y'].values.astype('int32')
        n = len(grp_s)
        for i in range(n - seq_len):
            labels.append(y_arr[i + seq_len])
            positions.append((key, i))
    return np.array(labels, dtype='int32'), positions

def build_sequences_from_indices(df_split, seq_len, feat_cols, scaler_, selected_pos):
    """Phase 2: construir X solo para indices seleccionados."""
    sel_set = set(map(tuple, [(k, i) for k, i in selected_pos]))
    groups = {key: grp.sort_values('window')
              for key, grp in df_split.groupby(ROUTE_KEYS, sort=False)}
    n_out = len(selected_pos)
    X_out = np.empty((n_out, seq_len, len(feat_cols)), dtype='float32')
    y_out = np.empty(n_out, dtype='int32')
    out_idx = 0
    for key, grp in groups.items():
        vals = scaler_.transform(grp[feat_cols].fillna(0).values.astype('float32'))
        y_arr = grp['y'].values.astype('int32')
        n = len(grp)
        for i in range(n - seq_len):
            if (key, i) in sel_set:
                X_out[out_idx] = vals[i:i + seq_len]
                y_out[out_idx] = y_arr[i + seq_len]
                out_idx += 1
    return X_out[:out_idx], y_out[:out_idx]

def build_sequences_val(df_split, seq_len, feat_cols, scaler_, max_samples):
    """Val sequences: secuencial (no necesita estratificar)."""
    groups = [(key, grp.sort_values('window'))
              for key, grp in df_split.groupby(ROUTE_KEYS, sort=False)
              if len(grp) >= seq_len + 1]
    total = sum(len(g) - seq_len for _, g in groups)
    n_out = min(total, max_samples)
    X_out = np.empty((n_out, seq_len, len(feat_cols)), dtype='float32')
    y_out = np.empty(n_out, dtype='int32')
    idx = 0
    for _, grp in groups:
        vals  = scaler_.transform(grp[feat_cols].fillna(0).values.astype('float32'))
        y_arr = grp['y'].values.astype('int32')
        n = len(grp)
        for i in range(n - seq_len):
            X_out[idx] = vals[i:i + seq_len]
            y_out[idx] = y_arr[i + seq_len]
            idx += 1
            if idx >= n_out:
                return X_out, y_out
    return X_out[:idx], y_out[:idx]

print('Phase 1: recolectando labels de train (sin X)...')
train_split = val_imp.copy()
print('  Recargando te_features_v5.parquet (solo split train)...')
df_reload = pq.read_table(IN_FEAT).to_pandas()
df_reload['y'] = df_reload.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_reload = df_reload.dropna(subset=['y'])
df_reload['y'] = df_reload['y'].astype(int)
ratio_cols_r = [c for c in feat_cols_v5 if 'ratio' in c]
other_nan_r  = [c for c in feat_cols_v5 if c not in ratio_cols_r]
df_reload[ratio_cols_r] = df_reload[ratio_cols_r].fillna(1.0)
df_reload[other_nan_r]  = df_reload[other_nan_r].fillna(0.0)
train_imp2 = df_reload[df_reload[SPLIT_COL]=='train'].reset_index(drop=True)
val_imp2   = df_reload[df_reload[SPLIT_COL]=='val'].reset_index(drop=True)
del df_reload
gc.collect()

all_labels_tr, all_positions_tr = build_sequences_labels_only(train_imp2, SEQ_LEN, FEAT_GRU)
print(f'  Total secuencias disponibles: {len(all_labels_tr):,}')
for c in range(4):
    n = (all_labels_tr==c).sum()
    print(f'  {LABEL_COLS[c]:<8}: {n:>7,} ({n/len(all_labels_tr):.2%})')

print(f'\nPhase 1b: StratifiedShuffleSplit -> {MAX_SEQ_TRAIN:,} secuencias estratificadas...')
sss = StratifiedShuffleSplit(n_splits=1, train_size=MAX_SEQ_TRAIN, random_state=RANDOM_STATE)
idx_strat, _ = next(sss.split(np.zeros(len(all_labels_tr)), all_labels_tr))
selected_positions = [all_positions_tr[i] for i in idx_strat]
selected_labels    = all_labels_tr[idx_strat]
print(f'  Composicion estratificada:')
for c in range(4):
    n = (selected_labels==c).sum()
    print(f'  {LABEL_COLS[c]:<8}: {n:>7,} ({n/MAX_SEQ_TRAIN:.2%})')

print('\nPhase 2: construyendo X para secuencias seleccionadas...')
X_tr_seq, y_tr_seq = build_sequences_from_indices(
    train_imp2, SEQ_LEN, FEAT_GRU, scaler, selected_positions)
print(f'  X_train_seq: {X_tr_seq.shape}')

print(f'\nConstruyendo secuencias val (max={MAX_SEQ_VAL:,})...')
X_vl_seq, y_vl_seq = build_sequences_val(val_imp2, SEQ_LEN, FEAT_GRU, scaler, MAX_SEQ_VAL)
print(f'  X_val_seq: {X_vl_seq.shape}')

del train_imp2, val_imp2, all_labels_tr, all_positions_tr
gc.collect()


class QWKLoss(keras.losses.Loss):
    """de la Torre et al. (2018, Pattern Recognition Letters 105:144-154)."""
    def __init__(self, n_classes=4, alpha=None, **kwargs):
        super().__init__(**kwargs)
        w = np.zeros((n_classes, n_classes), dtype=np.float32)
        for i in range(n_classes):
            for j in range(n_classes):
                w[i, j] = (i - j) ** 2 / (n_classes - 1) ** 2
        self.W = tf.constant(w, dtype=tf.float32)
        self.n = n_classes
        self.alpha = tf.constant(alpha, dtype=tf.float32) if alpha is not None else None

    def call(self, y_true, y_pred):
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=self.n)
        if self.alpha is not None:
            w_s      = tf.reduce_sum(y_true_oh * self.alpha, axis=1, keepdims=True)
            y_true_w = y_true_oh * w_s
        else:
            y_true_w = y_true_oh
        B   = tf.cast(tf.shape(y_true)[0], tf.float32)
        O   = tf.matmul(tf.transpose(y_true_w), y_pred) / B
        h_t = tf.reduce_sum(y_true_w, axis=0, keepdims=True) / B
        h_p = tf.reduce_mean(y_pred,   axis=0, keepdims=True)
        E   = tf.matmul(tf.transpose(h_t), h_p)
        return tf.reduce_sum(self.W * O) / (tf.reduce_sum(self.W * E) + 1e-9)


def build_gru_model(seq_len, n_features, n_classes=4):
    inp = keras.Input(shape=(seq_len, n_features))
    x   = layers.GRU(64, return_sequences=True,  dropout=0.2)(inp)
    x   = layers.GRU(32, return_sequences=False, dropout=0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out)

alpha_gru = sqrt_w_arr / sqrt_w_arr.sum()
tf.random.set_seed(RANDOM_STATE)
gru_v5 = build_gru_model(SEQ_LEN, len(FEAT_GRU))
gru_v5.summary()

gru_v5.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = QWKLoss(n_classes=4, alpha=alpha_gru),
    metrics   = ['accuracy'],
)

callbacks_v5 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=f'{OUT_DIR}/gru_v5_best.keras',
        monitor='val_loss', save_best_only=True, verbose=0),
]

t0 = time.time()
history_gru = gru_v5.fit(
    X_tr_seq, y_tr_seq,
    validation_data = (X_vl_seq, y_vl_seq),
    epochs     = 50,
    batch_size = 512,
    callbacks  = callbacks_v5,
    verbose    = 1,
)
elapsed = time.time() - t0
print(f'\nTiempo: {elapsed/60:.1f} min  |  Epochs: {len(history_gru.history["loss"])}')

y_prob_gru = gru_v5.predict(X_vl_seq, batch_size=2048, verbose=0)
y_pred_gru = np.argmax(y_prob_gru, axis=1)
resultados_v5['gru_v5'] = eval_model(y_vl_seq, y_pred_gru, 'GRU v5 (Estratificado) — Validacion')

gru_v5.save(f'{OUT_DIR}/gru_v5_model.keras')
print(f'Modelo guardado: {OUT_DIR}/gru_v5_model.keras')

r5 = resultados_v5['gru_v5']
r4 = resultados_v4_ref['gru_v4_val']
print('\nΔ GRU v4 -> GRU v5 (val):')
for m in ['qwk','recall_alta','macro_f1','bal_acc']:
    delta = r5[m] - r4.get(m, 0)
    print(f'  {m:<14}: v4={r4.get(m,0):.4f}  v5={r5[m]:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='gru_v5_stratified'):
    mlflow.log_param('model_type',          'gru')
    mlflow.log_param('seq_len',             SEQ_LEN)
    mlflow.log_param('n_features',          len(FEAT_GRU))
    mlflow.log_param('units',               '64-32')
    mlflow.log_param('loss',                'qwk_loss')
    mlflow.log_param('alpha_type',          'sqrt')
    mlflow.log_param('batch_size',          512)
    mlflow.log_param('max_seq_train',       MAX_SEQ_TRAIN)
    mlflow.log_param('sampling',            'stratified_D26')
    mlflow.log_param('epochs_run',          len(history_gru.history['loss']))
    mlflow.log_param('dataset_version',     'v5')
    for m in ['qwk','recall_alta','macro_f1','bal_acc','mae_ord','prec_alta']:
        mlflow.log_metric(f'val_{m}', r5[m])
    mlflow.log_metric('val_candidato', int(r5['candidato_crit']))
    mlflow.log_param('model_local_path',  f'{OUT_DIR}/gru_v5_model.keras')
print('GRU v5 (Estratificado) registrado en MLflow.')


Memoria liberada para GRU.
Phase 1: recolectando labels de train (sin X)...
  Recargando te_features_v5.parquet (solo split train)...
  Total secuencias disponibles: 2,121,638
  Normal  : 1,895,661 (89.35%)
  Baja    : 152,955 (7.21%)
  Media   :  36,362 (1.71%)
  Alta    :  36,660 (1.73%)

Phase 1b: StratifiedShuffleSplit -> 400,000 secuencias estratificadas...
  Composicion estratificada:
  Normal  : 357,396 (89.35%)
  Baja    :  28,837 (7.21%)
  Media   :   6,855 (1.71%)
  Alta    :   6,912 (1.73%)

Phase 2: construyendo X para secuencias seleccionadas...
  X_train_seq: (400000, 4, 100)

Construyendo secuencias val (max=200,000)...
  X_val_seq: (200000, 4, 100)


2026-08-06 15:31:15.219881: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4, 100)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 4, 64)          │        31,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,468 (165.89 KB)

 Trainable params: 42,468 (165.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.8122 - loss: 0.4219 - val_accuracy: 0.8328 - val_loss: 0.7363 - learning_rate: 0.0010
Epoch 2/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8252 - loss: 0.3781 - val_accuracy: 0.8155 - val_loss: 0.7254 - learning_rate: 0.0010
Epoch 3/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8229 - loss: 0.3786 - val_accuracy: 0.8296 - val_loss: 0.7102 - learning_rate: 0.0010
Epoch 4/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8337 - loss: 0.3765 - val_accuracy: 0.8029 - val_loss: 0.7098 - learning_rate: 0.0010
Epoch 5/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8257 - loss: 0.3745 - val_accuracy: 0.8295 - val_loss: 0.6931 - learning_rate: 0.0010
Epoch 6/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8303 - loss: 0.3735 - val_accuracy: 0.8301 - val_loss: 0.6972 - learning_rate: 0.0010
Epoch 7/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8364 - l

In [10]:

y_prob_xgb_ens = np.load(f'{OUT_DIR}/xgb_v5_val_proba.npy')
y_prob_lgb_ens = np.load(f'{OUT_DIR}/lgb_v5_val_proba.npy')

try:
    rf_v5_ens = rf_v5
    print('RF v5 en memoria.')
except NameError:
    print('Recargando RF v5 desde disco...')
    with open(f'{OUT_DIR}/rf_v5_model.pkl', 'rb') as f:
        rf_v5_ens = pickle.load(f)

X_val_reload = pq.read_table(IN_FEAT).to_pandas()
X_val_reload['y'] = X_val_reload.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
X_val_reload = X_val_reload.dropna(subset=['y'])
val_reload   = X_val_reload[X_val_reload[SPLIT_COL]=='val'].reset_index(drop=True)
ratio_c = [c for c in feat_cols_v5 if 'ratio' in c]
other_c = [c for c in feat_cols_v5 if c not in ratio_c]
val_reload[ratio_c] = val_reload[ratio_c].fillna(1.0)
val_reload[other_c] = val_reload[other_c].fillna(0.0)
X_val_sc_ens = scaler.transform(val_reload[feat_cols_v5].values.astype('float32'))
y_val_ens    = val_reload['y'].values.astype(int)
del X_val_reload
gc.collect()

y_prob_rf_ens = rf_v5_ens.predict_proba(X_val_sc_ens)

proba_rf_xgb = (y_prob_rf_ens + y_prob_xgb_ens) / 2
y_pred_rf_xgb = np.argmax(proba_rf_xgb, axis=1)
resultados_v5['ens_rf_xgb'] = eval_model(
    y_val_ens, y_pred_rf_xgb, 'Ensemble RF+XGB (soft-voting) — Validacion')

proba_3way = (y_prob_rf_ens + y_prob_xgb_ens + y_prob_lgb_ens) / 3
y_pred_3way = np.argmax(proba_3way, axis=1)
resultados_v5['ens_3way'] = eval_model(
    y_val_ens, y_pred_3way, 'Ensemble RF+XGB+LGB (soft-voting 3-way) — Validacion')

np.save(f'{OUT_DIR}/ens_3way_v5_val_proba.npy', proba_3way)
print(f'\nProbabilidades ensemble 3-way guardadas: {OUT_DIR}/ens_3way_v5_val_proba.npy')

for key in ['ens_rf_xgb', 'ens_3way']:
    r5 = resultados_v5[key]
    with mlflow.start_run(run_name=key + '_v5'):
        mlflow.log_param('model_type',      'ensemble_soft_voting')
        mlflow.log_param('components',      'rf+xgb' if '3way' not in key else 'rf+xgb+lgb')
        mlflow.log_param('dataset_version', 'v5')
        mlflow.log_param('d27_ensemble',    True)
        for m in ['qwk','recall_alta','macro_f1','bal_acc','mae_ord','prec_alta']:
            mlflow.log_metric(f'val_{m}', r5[m])
        mlflow.log_metric('val_candidato', int(r5['candidato_crit']))
    print(f'{key} registrado en MLflow.')


Recargando RF v5 desde disco...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   10.5s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:   10.9s finished



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Ensemble RF+XGB (soft-voting) — Validacion
  QWK=0.6415  MacroF1=0.6608  BalAcc=0.6902  MAEord=0.1628  RecAlta=0.6727  PrecAlta=0.5350
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Confusion matrix (filas=real, cols=pred):
            Normal   Baja    Media   Alta
  Normal    517,259  19,159   6,232   8,541
  Baja      15,098  40,022   2,186   1,339
  Media      4,140   1,866   6,900   1,863
  Alta       4,308     576   1,690  13,511

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Ensemble RF+XGB+LGB (soft-voting 3-way) — Validacion
  QWK=0.6399  MacroF1=0.6596  BalAcc=0.6955  MAEord=0.1657  RecAlta=0.6789  PrecAlta=0.5297
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
══════════════

In [11]:

W = 115
print('='*W)
print('  TABLA COMPARATIVA VALIDACION — v4 vs v5')
print('='*W)
print(f'  {"Modelo":<32}  {"QWK":>7}  {"RecAlta":>8}  {"MacroF1":>8}  {"BalAcc":>7}  {"H2?":>5}')
print('-'*W)

v4_rows = [
    ('Baseline v4 (persistencia)',   0.6149, 0.5598, None, None, False),
    ('RF v4 Full (113 feat)',        0.6257, 0.6466, None, None, False),
    ('RF v4 Sin-D12 (77 feat)',      0.6296, 0.6516, None, None, False),
    ('XGBoost v4',                   0.6141, 0.6941, None, None, False),
    ('GRU v4 (QWK Loss)',            0.4996, 0.0000, None, None, False),
]
for row in v4_rows:
    name, qwk, rec, f1, ba, cand = row
    f1s  = f'{f1:.4f}' if f1 else '  ---  '
    bas  = f'{ba:.4f}' if ba else '  ---  '
    cs   = '✅' if cand else '❌'
    print(f'  {name:<32}  {qwk:>7.4f}  {rec:>8.4f}  {f1s:>8}  {bas:>7}  {cs:>5}')

print(f'  {"":32}  {"---":>7}  {"---":>8}  {"---":>8}  {"---":>7}')

for key in ['baseline_v5','rf_v5','xgb_v5','lgb_v5','gru_v5','ens_rf_xgb','ens_3way']:
    if key not in resultados_v5:
        continue
    r = resultados_v5[key]
    name_map = {
        'baseline_v5' : 'Baseline v5 (persistencia)',
        'rf_v5'       : 'RF v5 (200 est, 100 feat)',
        'xgb_v5'      : 'XGBoost v5 (D19+D28)',
        'lgb_v5'      : 'LightGBM v5 (D25)',
        'gru_v5'      : 'GRU v5 (Estrat. D26)',
        'ens_rf_xgb'  : 'Ensemble RF+XGB (D27)',
        'ens_3way'    : 'Ensemble 3-way RF+XGB+LGB',
    }
    name = name_map.get(key, key)
    cs   = '✅ CANDIDATO' if r['candidato_crit'] else '❌'
    print(f'  {name:<32}  {r["qwk"]:>7.4f}  {r["recall_alta"]:>8.4f}'
          f'  {r["macro_f1"]:>8.4f}  {r["bal_acc"]:>7.4f}  {cs:>5}')

print('='*W)
print(f'  Floor H2: QWK >= 0.65 AND RecAlta >= 0.55 (Benchmarking v2.0 congelado)')
print('='*W)

candidatos_v5 = [k for k, r in resultados_v5.items() if r.get('candidato_crit', False)]
best_key = max(resultados_v5, key=lambda k: resultados_v5[k]['qwk'])
best_qwk = resultados_v5[best_key]['qwk']

print()
if candidatos_v5:
    print(f'✅ H2 VERIFICADA EN VAL v5')
    print(f'   Candidatos: {candidatos_v5}')
    print(f'   PROXIMO PASO: ejecutar 06_evaluacion_test_set_v5.ipynb (protocolo one-shot)')
    print(f'   IMPORTANTE: solo se evalua el test set UNA VEZ (Kaufman et al. 2012)')
else:
    gap = 0.65 - best_qwk
    print(f'❌ H2 NO VERIFICADA EN VAL v5 (0 candidatos)')
    print(f'   Mejor QWK v5 val: {best_qwk:.4f} (modelo: {best_key})')
    print(f'   Gap al floor: {gap:+.4f} QWK')
    print()
    print('   Acciones siguientes (LOG-17 §17.5):')
    print('   1. Regresion ordinal (Gutierrez et al. 2016, IEEE TNNLS) — alinea perdida con QWK')
    print('   2. Aumentar n_estimators RF (Breiman 2001) o LGB (Ke et al. 2017)')
    print('   3. Calibracion de probabilidades (Platt 2000) antes del ensemble')
    print('   4. Revisar HORIZON=2 (60 min) — aumentar H reduce ventaja de persistencia')

with open(f'{OUT_DIR}/resultados_v5_val.pkl', 'wb') as f:
    pickle.dump({k: {m: v for m, v in r.items() if m != 'confusion_matrix'}
                 for k, r in resultados_v5.items()}, f)
print(f'\nResultados v5 guardados: {OUT_DIR}/resultados_v5_val.pkl')


  TABLA COMPARATIVA VALIDACION — v4 vs v5
  Modelo                                QWK   RecAlta   MacroF1   BalAcc    H2?
-------------------------------------------------------------------------------------------------------------------
  Baseline v4 (persistencia)         0.6149    0.5598     ---      ---        ❌
  RF v4 Full (113 feat)              0.6257    0.6466     ---      ---        ❌
  RF v4 Sin-D12 (77 feat)            0.6296    0.6516     ---      ---        ❌
  XGBoost v4                         0.6141    0.6941     ---      ---        ❌
  GRU v4 (QWK Loss)                  0.4996    0.0000     ---      ---        ❌
                                        ---       ---       ---      ---
  Baseline v5 (persistencia)         0.6149    0.5598    0.6255   0.6254      ❌
  RF v5 (200 est, 100 feat)          0.6420    0.6726    0.6560   0.6720      ❌
  XGBoost v5 (D19+D28)               0.6232    0.6648    0.6511   0.6985      ❌
  LightGBM v5 (D25)                  0.6300    0.